# Overview of anomalies

In [1]:
import os
# Set environment variables to disable multithreading
# as users will probably want to set the number of cores
# to the max of their computer.
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from anomaly.constants import GALAXY_LINES
from anomaly.utils import specobjid_to_idx
from anomaly.utils import VelocityFilter
from anomaly.utils import AnomalyOverlapAnalyzer
from autoencoders.ae import AutoEncoder

from sdss.metadata import MetaData

meta = MetaData()

# Constants

In [3]:
mse_cols = ['mse', 'mse_filter_250', 'mse_97', 'mse_filter_250_97']
mse_rel_cols = ['mse_rel', 'mse_filter_250_rel', 'mse_97_rel', 'mse_filter_250_97_rel']

In [4]:
mse_family = ['mse', 'mse_97', 'mse_filter_250', 'mse_filter_250_97']
chi_sq_family = ['mse_rel', 'mse_97_rel', 'mse_filter_250_rel', 'mse_filter_250_97_rel']

# Custom functions

## IDs top anomalies

In [5]:
def get_ids(score, df, quantile=99, n_top=None, use_ntop=False):

    if use_ntop is False:
        
        quantile *= 0.01
        thresh = df[score].quantile(quantile)
        ids = set(df[df[score] > thresh].index)
        
    else:

        ids = set(
            df[score].sort_values(
                ascending=False
            ).iloc[:n_top].index
        )

    return ids

In [6]:
def top_unique_ids(scores_df, scores_list, quantile=99, n_top=None, use_ntop=False):

    ids_top_dict = {}

    for score in scores_list:

        ids_top_dict[score] = get_ids(
            score=score,
            df=scores_df,
            quantile=quantile,
            n_top=n_top,
            use_ntop=use_ntop
        )

    n_top = len(ids_top_dict[score])

    unique_ids_dict = AnomalyOverlapAnalyzer.get_unique_ids(
        ids_dict=ids_top_dict, score_list=scores_list
    )

    for score in scores_list:
        n_unique = len(unique_ids_dict[score])

        unique_pct = n_unique/n_top*100
        
        print(f"Unique to {score}:\n{n_unique} --> {unique_pct:.4f}%")

    return unique_ids_dict, ids_top_dict


In [7]:
def top_common_ids(scores_df, scores_list, quantile=99, n_top=None, use_ntop=False):

    ids_top_dict = {}

    for score in scores_list:

        ids_top_dict[score] = get_ids(
            score=score,
            df=scores_df,
            quantile=quantile,
            n_top=n_top,
            use_ntop=use_ntop
        )

    n_top = len(ids_top_dict[score])

    common_ids_dict = AnomalyOverlapAnalyzer.get_core_common_ids(
        ids_dict=ids_top_dict, score_list=scores_list
    )

    n_common = len(common_ids_dict)
    common_pct = n_common/n_top*100 
    print(f"N common:\n{n_common} -- > {common_pct:4f}%")

    return common_ids_dict, ids_top_dict

## Figures

In [8]:
def anomaly_plot(wave, specs, objids, ranks, save_to):

    fig, ax = plt.subplots(
        figsize=(10, 5)
    )

    for spec, objid, rank in zip(specs, objids, ranks):

        print(f'Rank {rank:03d}', end='\r')

        ax.clear()

        ax.plot(wave, spec, color="black", label=f'Rank: {rank}')

        ax.minorticks_on()
        ax.set_xlabel(r"$\lambda$ [nm]")
        ax.set_title(f"Object ID: {objid}")

        ax.legend(
            loc='upper left',
            frameon=False,
        )

        fig.savefig(
            f"{save_to}/{rank:03d}_{objid}.jpeg",
            bbox_inches='tight'
        )

    plt.close(fig)

# Config

## Directories

In [10]:
phd_dir = "/home/elom/phd"
thesis_dir = f"{phd_dir}/thesis"
data_dir = f"{phd_dir}/code"
spectra_dir = f"{data_dir}/spectra"
scores_dir = f"{data_dir}/scores"
models_dir = f"{data_dir}/models"
bin_id = 'bin_02'
#
ch_4_dir = f"{thesis_dir}/chapters/04_figures"

## Data

In [11]:
wave = np.load(f"{spectra_dir}/wave_spectra_imputed.npy")
wave_nm = wave*0.1

spectra = np.load(
    f"{spectra_dir}/spectra_imputed.npy",
    mmap_mode="r"
)

final_meta_df = pd.read_csv(
    f"{spectra_dir}/final_spec_n_z_warning_drop.csv.gz",
    index_col="specobjid",
)

idx_id_spec = np.load(
    f"{spectra_dir}/ids_imputing.npy",
    mmap_mode='r'
)

In [12]:
bin_id = f'{bin_id}'
score_df = pd.read_csv(
    f"{scores_dir}/{bin_id}/scores_{bin_id}.csv.gz",
    index_col='specobjid'
)
n_spec = score_df.shape[0]
n_top_1_pct = int(n_spec*0.01)
n_top_1_pct, n_spec

(1818, 181850)

## Append rank per score

In [13]:
rank = np.arange(score_df.shape[0])
score_rank_df = score_df.copy()
# score_rank_df
for col in score_df.columns:

    index_sorted = score_df.sort_values(
        by=col, ascending=False
    ).index

    score_rank_df.loc[index_sorted, f'rank_{col}'] = rank
    score_rank_df[f'rank_{col}'].astype(int)

In [14]:
score = 'mse_97'
score_rank_df[[score, f'rank_{score}']].sort_values(
    by=score, ascending=False
).head(10)

,mse_97,rank_mse_97
specobjid,,
1414177610681837568,9.461991,0.0
808477533984024576,9.285621,1.0
1959115916855764992,8.363419,2.0
1621402570222233600,8.360323,3.0
1506455496133994496,8.104469,4.0
3089488382624032768,7.921358,5.0
2399384299896858624,7.707820,6.0
2013174804214999040,7.604570,7.0
1001066349105014784,7.550252,8.0


## Model

In [15]:
ae_winner = AutoEncoder(
    reload=True,
    reload_from=f"{models_dir}/{bin_id}/winner_0021",
)

In [16]:
ae_winner.get_architecture_and_model_str()

['256_128_64_12_64_128_256', 'infoVae_rec_3776_alpha_0_lambda_13']

# Figures top anomalies

In [17]:
# ```python
n_top = 600
all_scores = mse_cols + mse_rel_cols
plt.ioff()

for score in all_scores:

    specids_top_1 = score_df[score].sort_values(
        ascending=False
    ).index.to_numpy()[:n_top]

    ranks = np.zeros(n_top).astype(int)

    specs_top_1 = np.empty((n_top, wave.size))

    for i, objid in enumerate(specids_top_1):

        spec_idx = specobjid_to_idx(
            objid, idx_id_spec
        )

        specs_top_1[i, :] = spectra[spec_idx, :]

        ranks[i] = i

    # -----------------------------------------------------------

    save_to = f"{scores_dir}/{bin_id}/figs/{score}"

    os.makedirs(save_to, exist_ok=True)

    anomaly_plot(
        wave_nm, specs=specs_top_1,
        objids=specids_top_1, ranks=ranks,
        save_to=save_to
    )
# ```

# No free lunch theorem

## IDs per score

In [18]:
ids_top_dict = {}

all_scores = mse_cols + mse_rel_cols

for score in all_scores:

    ids_top_dict[score] = get_ids(
        score=score,
        df=score_df.copy(),
        quantile=99,
        n_top=1000,
        use_ntop=False
    )

n_top_1 = len(ids_top_dict[score])
n_top_1

1819

# Distinct IDs

## MSE family

In [19]:
mse_unique_ids_dict, mse_ids_top_dict = top_unique_ids(
    scores_df=score_df.copy(),
    scores_list=mse_cols,
    quantile=99,
    n_top=None, use_ntop=False
)

Unique to mse:
499 --> 27.4327%
Unique to mse_filter_250:
312 --> 17.1523%
Unique to mse_97:
100 --> 5.4975%
Unique to mse_filter_250_97:
208 --> 11.4349%


In [21]:
score = 'mse'
unique_mse_ids = list(mse_unique_ids_dict[score])
score_rank_df.loc[
    unique_mse_ids, [score, f'rank_{score}']
].sort_values(by=score, ascending=False)

,mse,rank_mse
specobjid,,
2034670801378109440,18.662506,104.0
1462649032210933760,17.966851,110.0
2408458845228656640,17.529365,116.0
2222540217548564480,16.427243,142.0
2796771996883511296,15.501871,160.0
...,...,...
1445724245765154816,6.216540,1810.0
2034655133337413632,6.215901,1811.0
2277755224657520640,6.215720,1812.0


In [22]:
plt.ioff()

for score in mse_cols:

    specids = list(mse_unique_ids_dict[score])
    specids = np.array(specids, dtype=int)

    ranks = score_rank_df.loc[
        specids, f'rank_{score}'
    ].to_numpy().astype(int)

    specs = np.empty((len(specids), wave.size))

    for i, objid in enumerate(specids):

        spec_idx = specobjid_to_idx(
            objid, idx_id_spec
        )

        specs[i, :] = spectra[spec_idx, :]

    # -----------------------------------------------------------

    save_to = f"{scores_dir}/{bin_id}/figs/unique_mse/{score}"

    os.makedirs(save_to, exist_ok=True)

    anomaly_plot(
        wave_nm, specs=specs,
        objids=specids, ranks=ranks,
        save_to=save_to
    )

## Chi family

In [23]:
chi_unique_ids_dict, chi_ids_top_dict = top_unique_ids(
    scores_df=score_df.copy(),
    scores_list=mse_rel_cols,
    quantile=99,
    # n_top=1000, use_ntop=True
)

Unique to mse_rel:
304 --> 16.7125%
Unique to mse_filter_250_rel:
193 --> 10.6102%
Unique to mse_97_rel:
152 --> 8.3562%
Unique to mse_filter_250_97_rel:
221 --> 12.1495%


In [27]:
score = 'mse_97_rel'
unique_chi_ids = list(chi_unique_ids_dict[score])
score_rank_df.loc[
    unique_mse_ids, [score, f'rank_{score}']
].sort_values(by=score, ascending=False)

,mse_97_rel,rank_mse_97_rel
specobjid,,
920026290092795904,4.811891,302.0
2170780705942956032,4.799606,307.0
2382418286311663616,4.642682,457.0
3340568740008847360,4.531582,598.0
2112242176200042496,4.476053,711.0
...,...,...
2806896299465533440,4.207584,1799.0
454962548811261952,4.205850,1802.0
2486007400369252352,4.203233,1812.0


In [28]:
plt.ioff()

for score in mse_rel_cols:

    specids = list(chi_unique_ids_dict[score])
    specids = np.array(specids, dtype=int)

    ranks = score_rank_df.loc[
        specids, f'rank_{score}'
    ].to_numpy().astype(int)

    specs = np.empty((len(specids), wave.size))

    for i, objid in enumerate(specids):

        spec_idx = specobjid_to_idx(
            objid, idx_id_spec
        )

        specs[i, :] = spectra[spec_idx, :]

    # -----------------------------------------------------------

    save_to = f"{scores_dir}/{bin_id}/figs/unique_rse/{score}"

    os.makedirs(save_to, exist_ok=True)

    anomaly_plot(
        wave_nm, specs=specs,
        objids=specids, ranks=ranks,
        save_to=save_to
    )

## All scores

In [29]:
all_scores = mse_cols + mse_rel_cols

all_unique_ids_dict, all_ids_top_dict = top_unique_ids(
    scores_df=score_df.copy(),
    scores_list=all_scores,
    quantile=99,
    # n_top=1000, use_ntop=True
)

Unique to mse:
342 --> 18.8015%
Unique to mse_filter_250:
178 --> 9.7856%
Unique to mse_97:
63 --> 3.4634%
Unique to mse_filter_250_97:
115 --> 6.3222%
Unique to mse_rel:
38 --> 2.0891%
Unique to mse_filter_250_rel:
119 --> 6.5421%
Unique to mse_97_rel:
86 --> 4.7279%
Unique to mse_filter_250_97_rel:
184 --> 10.1154%


## Figures

In [30]:
all_scores = mse_cols + mse_rel_cols
plt.ioff()

for score in all_scores:

    specids = list(all_unique_ids_dict[score])
    specids = np.array(specids, dtype=int)

    ranks = score_rank_df.loc[
        specids, f'rank_{score}'
    ].to_numpy().astype(int)

    specs = np.empty((len(specids), wave.size))

    for i, objid in enumerate(specids):

        spec_idx = specobjid_to_idx(
            objid, idx_id_spec
        )

        specs[i, :] = spectra[spec_idx, :]

    # -----------------------------------------------------------

    save_to = f"{scores_dir}/{bin_id}/figs/unique_all/{score}"

    os.makedirs(save_to, exist_ok=True)

    anomaly_plot(
        wave_nm, specs=specs,
        objids=specids, ranks=ranks,
        save_to=save_to
    )

# Common IDs

## MSE family

In [37]:
mse_common_ids_dict, mse_ids_top_dict = top_common_ids(
    scores_df=score_df.copy(), scores_list=mse_cols,
    quantile=99,
    n_top=None, use_ntop=False
)

N common:
561 -- > 30.841121%


In [38]:
plt.ioff()

specids = list(mse_common_ids_dict)
specids = np.array(specids, dtype=int)

ranks = score_rank_df.loc[
    specids, f'rank_mse'
].to_numpy().astype(int)

specs = np.empty((len(specids), wave.size))

for i, objid in enumerate(specids):

    spec_idx = specobjid_to_idx(
        objid, idx_id_spec
    )

    specs[i, :] = spectra[spec_idx, :]

# -----------------------------------------------------------

save_to = f"{scores_dir}/{bin_id}/figs/common_se"

os.makedirs(save_to, exist_ok=True)

anomaly_plot(
    wave_nm, specs=specs,
    objids=specids, ranks=ranks,
    save_to=save_to
)

## Chi Family

In [39]:
chi_common_ids_dict, chi_ids_top_dict = top_common_ids(
    scores_df=score_df.copy(), scores_list=mse_rel_cols,
    quantile=99,
    n_top=None, use_ntop=False
)

N common:
757 -- > 41.616273%


In [41]:
plt.ioff()

specids = list(chi_common_ids_dict)
specids = np.array(specids, dtype=int)

ranks = score_rank_df.loc[
    specids, 'rank_mse_rel'
].to_numpy().astype(int)

specs = np.empty((len(specids), wave.size))

for i, objid in enumerate(specids):

    spec_idx = specobjid_to_idx(
        objid, idx_id_spec
    )

    specs[i, :] = spectra[spec_idx, :]

# -----------------------------------------------------------

save_to = f"{scores_dir}/{bin_id}/figs/common_rse"

os.makedirs(save_to, exist_ok=True)

anomaly_plot(
    wave_nm, specs=specs,
    objids=specids, ranks=ranks,
    save_to=save_to
)

## All scores

In [42]:
all_scores = mse_cols + mse_rel_cols 
all_common_ids_dict, all_ids_top_dict = top_common_ids(
    scores_df=score_df.copy(), scores_list=all_scores,
    quantile=99,
    n_top=None, use_ntop=False
)

N common:
320 -- > 17.592084%


### Figures

In [43]:
all_scores = mse_cols + mse_rel_cols
plt.ioff()

specids = list(all_common_ids_dict)
specids = np.array(specids, dtype=int)

ranks = score_rank_df.loc[
    specids, 'rank_mse'
].to_numpy().astype(int)

specs = np.empty((len(specids), wave.size))

for i, objid in enumerate(specids):

    spec_idx = specobjid_to_idx(
        objid, idx_id_spec
    )

    specs[i, :] = spectra[spec_idx, :]

# -----------------------------------------------------------

save_to = f"{scores_dir}/{bin_id}/figs/common_all"

os.makedirs(save_to, exist_ok=True)

anomaly_plot(
    wave_nm, specs=specs,
    objids=specids, ranks=ranks,
    save_to=save_to
)